In [15]:
import numpy as np
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [16]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def h_3(x,alfa):
    return(min(1,x/(1-alfa)))

def riskcalc(a,R,p,alfa,r_f):
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    print(rank)
    extra = 0
    if np.min(x) < 0:
        extra = np.min(x)
        x = x - np.min(x)
    N = len(p)
    risk = h_3(p[rank[0]],alfa)*x[rank[0]]
    for i in range(2,N+1):
        z1 = sum(p[rank[0:i]])
        z2 = sum(p[rank[0:i-1]])
        print(p[rank[0:i]])
        print(p[rank[0:i-1]])
        risk = risk + (h_3(z1,alfa)-h_3(z2,alfa))*x[rank[i-1]]
    risk = risk + extra - (1-sum(a))*r_f
    return(risk)

In [75]:
def makeset (A, B):    # we assume A is non-empty
    N = len(A)
    added = []
    for i in range(N):
        new = B[0:i+1]
        N_sets = len(A[i])
        for k in range(N_sets-1):
            if len(np.intersect1d(A[i][k],new))==len(new):
                break
            if k == N_sets-2:
                A[i].append(new)
                added.append(new)
    return(A,added)

def countsets(sets):
    m = len(sets)
    count = 0
    for k in range(m):
        count = count + len(sets[k])
    return(count)

def convertlist(sets):
    Output = []
    for temp in sets:
        for elem in temp:
            Output.append(elem)
    return(Output)

def solvenominal (sets,p,R,r,m,r_f,c):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable((M,N))
    lbda = cp.Variable(M, nonneg = True)
    a = cp.Variable(I)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N, nonneg = True)
    z2 = 0
    z4 = 0
    constraints = []
    for i in range(N):
        lbdasum = 0
        for j in range(M):
            if i in sets[j]:
                constraints.append(v[j][i] >= 0)
                lbdasum = lbdasum + lbda[j]
            else:
                constraints.append(v[j][i] >= 0)
        constraints.append((-R @ a)[i] - lbdasum - beta <= 0)
        z4 = z4 + p[i]*t[i]
        constraints.append(-alpha + cp.sum(v[0:M:1,i]) + cp.kl_div(gamma, t[i]) + gamma - t[i] <= 0)
    for j in range(M):
        z1 = -cp.min(v[j,sets[j]])*(1-m)+lbda[j]
        z2 = z2 + cp.pos(z1)
    constraints.append(cp.abs(a)<=20)
    constraints.append(alpha + beta + gamma * (r-1) - (1-cp.sum(a))*r_f + z4 + z2 <= c)
    
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value,v.value,a.value,lbda.value,alpha.value,beta.value,gamma.value)
    
    
def robustcheck(a,R,r,c,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    if np.min(x) < 0:
        c = c - np.min(x)
        x = x - np.min(x)
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    print(c)
    print(prob.value - (1-np.sum(a))*r_f)
    return(prob.value,q.value,q_b.value,(prob.value - (1-np.sum(a))*r_f) <= c)
    
                
    

In [76]:
np.random.seed(10)
N=4
x=np.array([1,2,3])
psets = list(powerset(list(range(N))))
for i in range(1,len(psets)):
    psets[i] = list(psets[i])
psets = psets[1:(len(psets))]
psets

[[0],
 [1],
 [2],
 [3],
 [0, 1],
 [0, 2],
 [0, 3],
 [1, 2],
 [1, 3],
 [2, 3],
 [0, 1, 2],
 [0, 1, 3],
 [0, 2, 3],
 [1, 2, 3],
 [0, 1, 2, 3]]

In [77]:
r = 1
m = 0.2
r_f = 0.001
c = 0.001
p = np.random.rand(4)
p = p/sum(p)
R = np.random.rand(4,5)*2-1
sets =psets               #[[0],[1],[0,1],[1,3],[0,3],[0,1,2],[1,3,2],[0,1,3],[0,1,2,3]]
solvenominal (sets,p,R,r,m,r_f,c)

(11.163979845235717,
 array([[ 6.98387153e-01,  3.40935205e-08,  8.44635402e-08,
          7.19515807e-08],
        [ 3.87779753e-08,  1.47961991e+01, -0.00000000e+00,
         -0.00000000e+00],
        [-0.00000000e+00, -0.00000000e+00,  2.04398670e-06,
         -0.00000000e+00],
        [-0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
          2.10151292e-07],
        [ 1.15473045e+00,  1.15473045e+00, -0.00000000e+00,
         -0.00000000e+00],
        [ 3.06252621e-06, -0.00000000e+00,  3.11046772e-06,
         -0.00000000e+00],
        [ 2.21395143e-07, -0.00000000e+00, -0.00000000e+00,
          2.71539891e-07],
        [-0.00000000e+00,  3.06423415e-06,  3.11672764e-06,
         -0.00000000e+00],
        [-0.00000000e+00,  2.20378428e-07, -0.00000000e+00,
          2.76634657e-07],
        [-0.00000000e+00, -0.00000000e+00,  2.85033563e-07,
          2.63501966e-07],
        [ 3.03197780e-06,  3.02690934e-06,  3.08015510e-06,
         -0.00000000e+00],
        [ 2.57210910e-

In [81]:
a=solvenominal (sets,p,R,r,m,r_f,c)[2]           #np.array([1/5,1/5,1/5,1/5,1/5])
print(R.dot(a).dot(p))
print(-R.dot(a))
print(1-sum(a))
print(a)

11.1166891733205
[ -4.60317234   6.67507723  -6.08566259 -22.57646542]
47.29067191521686
[  1.5351075  -19.99999976 -19.99999691 -19.99999987  12.17421713]


In [79]:
robustcheck(a,R,r,c,p,m,r_f)

22.577465417314013
22.577463969048434


(22.62475464096365,
 array([0.31059165, 0.34846823, 0.156271  , 0.18466913]),
 array([ 0.38823956,  0.43558528,  0.17617516, -0.        ]),
 True)

In [63]:
q = robustcheck(a,R,r,c,p,m,r_f)[1]
phi = 0
for i in range(len(p)):
    phi = phi + p[i]*np.log(p[i]/q[i])
phi

11.290697639703104
11.290697571748739


0.40891935230535825

In [10]:
riskcalc(a,R,p,m,r_f)

[1 0 2 3]
[0.00954321 0.35470769]
[0.00954321]
[0.00954321 0.35470769 0.2913962 ]
[0.00954321 0.35470769]
[0.00954321 0.35470769 0.2913962  0.3443529 ]
[0.00954321 0.35470769 0.2913962 ]


-10.258833604974658

In [11]:
def robustcheck2(a,R,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    extra = 0
    if np.min(x) < 0:
        extra = np.min(x)
        x = x - np.min(x)
    q_b = cp.Variable(N, nonneg = True)
    constraints = []
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = p[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    print(c)
    print(prob.value - (1-np.sum(a))*r_f+extra)
    return(prob.value,q_b.value)

In [12]:
robustcheck2(a,R,p,m,r_f)

0.001
-10.258833604974658


(23.30108300108709, array([0.44338462, 0.01192901, 0.36424525, 0.18044112]))

In [214]:
a

array([ 30., -30., -30., -30.,  30.])